# Monitoring as a Post-5G library stage

Load your la<YOUR_PASSWORD> deployment models and monitoring configuration, inspect the plan,
then enable deployment. See ../../MONITORING.md for prerequisites and configuration.
`RUN_LIVE = False` prepares the plan without contacting infrastructure.


In [ ]:
import asyncio
import json
from datetime import datetime, UTC
from pathlib import Path

from orchestrator.models.monitoring import MonitoringDeployment
from orchestrator.deployment import BaseManagement, MonitoringManagement, UEManagement
from orchestrator.artifacts import ExperimentRun
from orchestrator.experiments import run_legacy_experiment

RUN_LIVE = False
RUN_NATIVE_PING = False
RUN_LEGACY = False
REMOVE_MONITORING = False


In [ ]:
previous = BaseManagement(
    metamodel_file="metamodel.yaml",
    deployment_model_file="deployment-model.yaml",
)
previous.deployment_model.monitoring = MonitoringDeployment.from_yaml_file("monitoring.yaml")
monitoring = MonitoringManagement(preceding=previous)
plan = monitoring.prepare()
print(json.dumps({"control_node": plan["control_node"], "namespace": plan["namespace"],
                  "targets": plan["targets"], "notes": plan["notes"]}, indent=2))


## Deploy after inspecting the plan

The live preflight checks the credentials Secret, scheduling node, exact pod selectors,
containers and packet interfaces. Keep the stack in a separate trial namespace.
Deploy monitoring after core/RAN are running and before the UE attachment events you want to measure.
`wait_ready()` checks workload rollouts; verify the expected time series and UE mappings before measurements.


In [ ]:
if RUN_LIVE:
    try:
        await monitoring.deploy()
    finally:
        # Preserve partial state too, so removal can be retried after a failure.
        monitoring.output(metamodel_file="metamodel-after-monitoring.yaml",
                          deployment_model_file="deployment-after-monitoring.yaml")
else:
    print("Offline preview: deployment skipped.")


## Native experiment sections and artifacts

Set a Prometheus URL reachable from the notebook and a query profile for your
measurements. Enable the optional UE ping to record its timeline and metrics.


In [ ]:
PROMETHEUS_URL = None  # URL reachable from this notebook.
QUERIES_FILE = None    # Path to your experiment query profile.
NATIVE_UE = None       # Deployed containerized UE component name.
PING_DESTINATION = None
UE_INTERFACE = "uesimtun0"  # Verify against your selected UE.

if RUN_LIVE and RUN_NATIVE_PING:
    if not all([PROMETHEUS_URL, QUERIES_FILE, NATIVE_UE, PING_DESTINATION]):
        raise ValueError("Set the Prometheus URL, query profile, UE and destination")
    out = Path("results") / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    ues = UEManagement(preceding=monitoring)
    async with ExperimentRun(out, name="ue-ping", prometheus_url=PROMETHEUS_URL,
                             queries_file=Path(QUERIES_FILE)) as run:
        with run.section("baseline"):
            await asyncio.sleep(1)
        with run.section("ping"):
            result = await ues.run_command(
                components=[NATIVE_UE],
                command=["ping", "-I", UE_INTERFACE, "-c", "5", PING_DESTINATION],
            )
            (out / "ping.json").write_text(json.dumps(result, indent=2))
    print(f"Artifacts: {out.resolve()}")


## Retain costly scenario playbooks deliberately

This bridge calls `Miscellaneous.run_playbook()`. It requires a clean **develop** checkout
and an inventory with the groups expected by the chosen old scenario. Review its variables:
physical UEs use SSH/wwan0 in some scenarios, and many old playbooks still conflate core type
with namespace. The bridge does not silently claim those scenarios have been ported.

Pod logs and pcaps remain available through the legacy experiment collector in this trial.


In [ ]:
if RUN_LIVE and RUN_LEGACY:
    await run_legacy_experiment(
        monitoring,
        legacy_root="/path/to/clean/5g_ansible-develop",
        scenario="scenarios/experiment_examples/two_ue_iperf_40m.yml",
        inventory="/path/to/reviewed/legacy-hosts.ini",
        artifact_profile="configs/artifacts/profiles/default_5g_observability.yml",
        variables={
            "core": "open5gs", "ran": "oai",
            "experiment_prometheus_url": PROMETHEUS_URL,
            "experiment_pod_log_namespaces": ["open5gs-core", "oai-ran", "monitoring"],
        },
    )
else:
    print("Legacy scenario execution disabled.")


## Remove the trial

Removal stops the controller, runs a cleanup Job for owned sniffers/probes, then removes
its addon resources and Helm releases. It retains the namespace, credentials Secret,
Prometheus CRDs, and network-function pods. Terminated ephemeral-container records remain
until those pods are replaced. A cleanup failure is reported and retains state for retry.


In [ ]:
if RUN_LIVE and REMOVE_MONITORING:
    try:
        await monitoring.remove()
    finally:
        monitoring.output(deployment_model_file="deployment-after-monitoring.yaml")
else:
    print("No monitoring resources removed.")
